In [15]:
# Install CLEAN Compatible Packages

In [1]:
!pip install -q transformers==4.41.2 \
sentence-transformers==2.7.0 \
accelerate \
faiss-cpu \
rank-bm25 \
pypdf \
python-pptx \
python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 141.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 20.2 MB/s eta 0:00:00


In [16]:
# Import Libraries

In [2]:
import os
import torch
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [17]:
# Load Embedding Model (Free, Local)

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded


In [18]:
# Load LLM (Stable Chat Model)

In [5]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7
)

print("✅ LLM loaded successfully")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ LLM loaded successfully


In [19]:
# Upload Multiple Documents (PDF, PPT, DOCX)

In [6]:
from google.colab import files
uploaded = files.upload()

Saving Regularisation.pptx to Regularisation.pptx


In [20]:
# Extract Text From All File Types

In [7]:
from pypdf import PdfReader
from docx import Document
from pptx import Presentation

documents = []

for file in uploaded.keys():
    if file.endswith(".pdf"):
        reader = PdfReader(file)
        for i, page in enumerate(reader.pages):
            documents.append({
                "content": page.extract_text(),
                "source": file,
                "page": i+1
            })

    elif file.endswith(".docx"):
        doc = Document(file)
        text = "\n".join([p.text for p in doc.paragraphs])
        documents.append({
            "content": text,
            "source": file,
            "page": 1
        })

    elif file.endswith(".pptx"):
        prs = Presentation(file)
        for i, slide in enumerate(prs.slides):
            text = ""
            for shape in slide.shapes:
                if hasattr(shape, "text"):
                    text += shape.text + "\n"
            documents.append({
                "content": text,
                "source": file,
                "page": i+1
            })

print("✅ Documents processed:", len(documents))

✅ Documents processed: 10


In [21]:
# Create Embeddings + FAISS Index

In [8]:
texts = [doc["content"] for doc in documents]

embeddings = embedding_model.encode(texts)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("✅ FAISS index created")

✅ FAISS index created


In [22]:
# BM25 Setup

In [9]:
tokenized_texts = [text.split() for text in texts]
bm25 = BM25Okapi(tokenized_texts)

print("✅ BM25 ready")

✅ BM25 ready


In [23]:
# Hybrid Search Function

In [10]:
def hybrid_search(query, top_k=3):
    query_embedding = embedding_model.encode([query])

    _, faiss_indices = index.search(np.array(query_embedding), top_k)
    bm25_scores = bm25.get_scores(query.split())
    bm25_top = np.argsort(bm25_scores)[-top_k:]

    combined = list(set(faiss_indices[0]) | set(bm25_top))

    results = [documents[i] for i in combined]
    return results

In [24]:
# Add Memory + Chat Function

In [11]:
chat_history = []

def ask_bot(query):
    results = hybrid_search(query)

    context = ""
    sources = []

    for res in results:
        context += res["content"] + "\n"
        sources.append(f"{res['source']} (Page {res['page']})")

    memory = ""
    for chat in chat_history:
        memory += f"User: {chat['user']}\nAssistant: {chat['bot']}\n"

    prompt = f"""
You are a helpful AI assistant.

Conversation History:
{memory}

Context:
{context}

Question:
{query}

Answer:
"""

    output = pipe(prompt)[0]["generated_text"]
    response = output.split("Answer:")[-1].strip()

    chat_history.append({"user": query, "bot": response})

    return response, sources

In [25]:
# Test It

In [12]:
response, sources = ask_bot("What is the main topic?")
print("Answer:\n", response)
print("\nSources:\n", sources)

Answer:
 Sure! Suppose we have a neural network with two layers: input layer, hidden layer, and output layer. In this case, we want to regularise the weights of the input layer to prevent overfitting. To do so, we add a regularisation term to

Sources:
 ['Regularisation.pptx (Page 1)', 'Regularisation.pptx (Page 2)', 'Regularisation.pptx (Page 3)', 'Regularisation.pptx (Page 9)']


In [26]:
# CONVERSATION FLOW TEST....

In [13]:
response, sources = ask_bot("What is the main topic of the document?")
print(response)

There are two main types of regularisation techniques in Deep Learning:
1. L2 Regularisation
2. L1 Regularisation

L2 Regularisation:
This technique penalises the weights of the model by adding a penalty term to the loss function. The penalty term is usually in the form of a L2 norm, which is the square of the Euclidean distance between each weight and its corresponding value in the input layer.


In [14]:
response, sources = ask_bot("Explain it in simpler words.")
print(response)

Regularization is a technique used to make the model more stable and prevent it from overfitting. In simpler terms, it helps to regularize the model by adding regularization terms to the loss function. Regularization in deep learning helps to prevent the model from becoming too complex and ensures that it can learn general patterns instead of noise. It is a crucial technique that is used in image and speech tasks. It is used most often in image and speech tasks. It is effective in reducing overfitting, improving test accuracy, and making the model more stable.
